# Prepare your photos for Meandering Photo Lines

Place original photos in `originalSourceFiles` at the project root. Subfolders are included. Run the cells below to read EXIF metadata, generate the photo CSV and export matching WebP files.

JPEG, HEIC/HEIF, PNG, TIFF and WebP are supported when readable by Pillow and pillow-heif. For Live Photos, only the still image is imported. RAW files and videos are not imported.

The originals are never modified. `L` keeps the full, orientation-corrected image dimensions; `S` and `XS` keep the aspect ratio. EXIF metadata is not embedded in the exported WebPs. Coordinates and dates are stored in the CSV instead.

Please note that this notebook does not group your images by series and does not generate keywords.


In [ ]:
# Run once to install the required packages. Restart the kernel afterwards if needed.
%pip install pillow pillow-heif reverse_geocoder pycountry


In [ ]:
from pathlib import Path
from datetime import datetime
from io import BytesIO
import csv
import hashlib
import re
import math

from PIL import Image, ImageOps, ImageCms
from pillow_heif import register_heif_opener
import reverse_geocoder
import pycountry
from IPython.display import display

register_heif_opener()


## Settings

The default output paths match the project config. Because the project may already contain a dataset, replacing `data.csv` requires setting `overwriteData = True`. Alternatively, change both output paths to a separate folder for a first test.

`webpQuality` controls lossy WebP compression. The large image keeps its original pixel dimensions, but it is recompressed.


In [ ]:
######################Paths and export settings

processingFolder = Path.cwd().resolve()
if processingFolder.name == "0_originalImageProcessing":
    projectFolder = processingFolder.parent.parent
elif processingFolder.name == "dataProcessing":
    projectFolder = processingFolder.parent
else:
    projectFolder = processingFolder

inputFolder = projectFolder / "originalSourceFiles"
outputDataFile = projectFolder / "data.csv"
outputImageFolder = projectFolder / "assets/img/photos"

smallSize = 640
extraSmallSize = 160
webpQuality = 90
overwriteData = True
photoLimit = None  # For a test, use e.g. 10. None imports all supported photos.

imageExtensions = {".jpg", ".jpeg", ".heic", ".heif", ".png", ".tif", ".tiff", ".webp"}


## Read EXIF metadata

The capture date comes from `DateTimeOriginal`, then `DateTimeDigitized`, then the general EXIF `DateTime` field. Filesystem modification dates are not used. EXIF fractional seconds and the corresponding UTC offset are retained when present. Without an offset, the date remains local camera time; the notebook does not guess a time zone.

Photos without a readable date or GPS coordinates are skipped and listed below. Enable location recording on the phone and export originals with metadata to preserve this information.


In [ ]:
######################EXIF helpers

def exifText(value):
    if isinstance(value, bytes):
        value = value.decode("ascii", errors="replace")
    return str(value or "").strip("\x00 ")

def captureDate(exif):
    details = exif.get_ifd(0x8769)
    # Date, fractional seconds and UTC offset must refer to the same EXIF date field.
    dateTags = [(36867, 37521, 36881), (36868, 37522, 36882), (306, 37520, 36880)]
    for dateTag, fractionTag, offsetTag in dateTags:
        value = exifText(details.get(dateTag) or exif.get(dateTag))
        if value == "":
            continue
        try:
            date = datetime.strptime(value, "%Y:%m:%d %H:%M:%S")
        except ValueError:
            continue
        result = date.isoformat(timespec="seconds")
        fraction = exifText(details.get(fractionTag) or exif.get(fractionTag))
        if fraction.isdigit():
            result += "." + fraction
        offset = exifText(details.get(offsetTag) or exif.get(offsetTag))
        if re.fullmatch(r"[+-]\d{2}:\d{2}", offset):
            result += offset
        return result
    raise ValueError("No readable EXIF capture date")

def gpsCoordinates(exif):
    gps = exif.get_ifd(0x8825)
    if any(tag not in gps for tag in [1, 2, 3, 4]):
        raise ValueError("No EXIF GPS coordinates")
    latitudeParts = [float(value) for value in gps[2]]
    longitudeParts = [float(value) for value in gps[4]]
    latitude = latitudeParts[0] + latitudeParts[1] / 60 + latitudeParts[2] / 3600
    longitude = longitudeParts[0] + longitudeParts[1] / 60 + longitudeParts[2] / 3600
    if exifText(gps[1]).upper() == "S":
        latitude = -latitude
    if exifText(gps[3]).upper() == "W":
        longitude = -longitude
    if not math.isfinite(latitude) or not math.isfinite(longitude) or abs(latitude) > 90 or abs(longitude) > 180:
        raise ValueError("Invalid EXIF GPS coordinates")
    return latitude, longitude

def photoId(path):
    # Relative paths keep IDs stable across runs and separate identical filenames in different folders.
    relativePath = path.relative_to(inputFolder).as_posix()
    return "photo-" + hashlib.sha256(relativePath.encode("utf-8")).hexdigest()[:16]

def prepareImage(source):
    image = ImageOps.exif_transpose(source)
    profile = image.info.get("icc_profile")
    if profile:
        sourceProfile = ImageCms.ImageCmsProfile(BytesIO(profile))
        targetProfile = ImageCms.createProfile("sRGB")
        image = ImageCms.profileToProfile(image, sourceProfile, targetProfile, outputMode="RGB")
    else:
        image = image.convert("RGB")
    # Keep pixel colors, but do not copy location or camera metadata into exported images.
    image.info.clear()
    return image


In [ ]:
######################Scan the source photos

if not inputFolder.is_dir():
    raise FileNotFoundError(f"Create this folder and add your original photos: {inputFolder}")

sourceFiles = sorted(path for path in inputFolder.rglob("*") if path.is_file()
                     and path.suffix.lower() in imageExtensions and not path.name.startswith("."))
if photoLimit is not None:
    sourceFiles = sourceFiles[:photoLimit]

photos = []
skippedPhotos = []
for index, path in enumerate(sourceFiles, start=1):
    try:
        with Image.open(path) as image:
            exif = image.getexif()
            date = captureDate(exif)
            latitude, longitude = gpsCoordinates(exif)
        photos.append({"path": path, "id": photoId(path), "date": date, "lat": latitude, "lon": longitude})
    except (OSError, ValueError, TypeError, KeyError, IndexError, ZeroDivisionError) as error:
        skippedPhotos.append({"file": str(path.relative_to(inputFolder)), "reason": str(error)})
    if index % 100 == 0:
        print(f"Read metadata from {index}/{len(sourceFiles)} photos")

print(f"{len(photos)} photos ready; {len(skippedPhotos)} skipped")
if skippedPhotos:
    display(skippedPhotos)
if not photos:
    raise ValueError("No photos with a readable EXIF date and GPS coordinates were found.")


## Add place names locally

EXIF generally stores coordinates rather than country or city names. The offline `reverse_geocoder` database finds the nearest town and its administrative region and country. No photo coordinates are sent to an online service.

These names are approximate: the nearest town may differ from the actual location, especially near borders or in remote areas. Exact GPS coordinates are preserved from EXIF. This does not perform automatic keyword detection or series grouping.

Sources: [reverse_geocoder](https://github.com/thampiman/reverse-geocoder), [Pillow EXIF support](https://pillow.readthedocs.io/en/stable/reference/Image.html), [pillow-heif](https://pillow-heif.readthedocs.io/en/latest/).


In [ ]:
######################Look up country and place labels

coordinates = list(dict.fromkeys((photo["lat"], photo["lon"]) for photo in photos))
places = reverse_geocoder.search(coordinates, mode=1)
placeLookup = dict(zip(coordinates, places))
for photo in photos:
    place = placeLookup[(photo["lat"], photo["lon"])]
    country = pycountry.countries.get(alpha_2=place["cc"])
    photo["country"] = country.name if country else place["cc"]
    photo["city"] = place["name"]
    photo["region"] = place["admin1"]
    photo["district"] = place["admin2"]
    photo["place_short"] = ", ".join(part for part in [photo["city"], photo["country"]] if part)

# UTC offsets are used where available. Unzoned camera dates remain local wall-clock time.
def dateSortKey(photo):
    date = datetime.fromisoformat(photo["date"])
    if date.tzinfo is not None:
        from datetime import timezone
        date = date.astimezone(timezone.utc).replace(tzinfo=None)
    return date, photo["id"]

photos.sort(key=dateSortKey)
display([{key: photo[key] for key in ["id", "date", "country", "place_short"]} for photo in photos[:10]])


## Export the CSV and WebP images

The CSV includes the project's required columns. `series_id` and `keywords` are deliberately empty; without a series ID, the visualization treats each photo as a separate series. `orig_width` and `orig_height` describe the orientation-corrected image.

Existing WebPs with matching IDs are regenerated, so edits to the source photos or quality setting take effect. Existing `data.csv` is replaced only when `overwriteData = True`. Other image files are left in place.


In [ ]:
######################Write the dataset and image sizes

if outputDataFile.exists() and overwriteData == False:
    raise FileExistsError(f"{outputDataFile} already exists. Set overwriteData = True or choose a different outputDataFile.")

for size in ["L", "S", "XS"]:
    (outputImageFolder / size).mkdir(parents=True, exist_ok=True)
outputDataFile.parent.mkdir(parents=True, exist_ok=True)

columns = ["id", "orig_width", "orig_height", "lat", "lon", "date", "series_id", "keywords",
           "district", "city", "region", "country", "place_short", "source_file"]
records = []
for index, photo in enumerate(photos, start=1):
    with Image.open(photo["path"]) as source:
        image = prepareImage(source)
    width, height = image.size
    for size, limit in [("L", None), ("S", smallSize), ("XS", extraSmallSize)]:
        variant = image.copy()
        if limit is not None:
            variant.thumbnail((limit, limit), Image.Resampling.LANCZOS)
        variant.save(outputImageFolder / size / (photo["id"] + ".webp"),
                     "WEBP", quality=webpQuality, method=4)
        variant.close()
    image.close()

    records.append({
        "id": photo["id"], "orig_width": width, "orig_height": height,
        "lat": photo["lat"], "lon": photo["lon"], "date": photo["date"],
        "series_id": "", "keywords": "", "district": photo["district"], "city": photo["city"],
        "region": photo["region"], "country": photo["country"], "place_short": photo["place_short"],
        "source_file": photo["path"].relative_to(inputFolder).as_posix(),
    })
    if index % 25 == 0 or index == len(photos):
        print(f"Exported {index}/{len(photos)} photos")

with outputDataFile.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=columns)
    writer.writeheader()
    writer.writerows(records)

print(f"CSV: {outputDataFile}")
print(f"Images: {outputImageFolder} ({len(records) * 3} WebP files)")
